In [ ]:
import numpy as np

from mbloodmoon.images import upscale, downscale
import mbloodmoon as bm
from IROS_pipeline import _handle_dirpaths

mask_FITS = "wfm_mask.fits"

skyfield = "GalacticCenter"
data_FITS = "20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb"

mask_file, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
)

wfm = bm.codedmask(mask_file, upscale_x=1, upscale_y=1)

In [6]:
sky = np.ones(wfm.sky_shape)

down_sky = downscale(sky, *(5, 3))
up_sky = upscale(down_sky, *(5, 3))


sky.shape, up_sky.shape, down_sky.shape, int(sky.sum()), int(up_sky.sum()), int(down_sky.sum())

((1033, 1671), (1030, 1671), (206, 557), 1726143, 1726143, 1726143)

In [15]:
wfm2 = bm.codedmask(mask_file, upscale_x=3, upscale_y=5)

wfm2.sky_shape

(5163, 5015)

In [1]:
import numpy as np
import mbloodmoon as bm

mask_path = "/mnt/d/PhD_AASS/Coding/Images_fits/wfm_mask.fits"
wfm = bm.codedmask(mask_path, upscale_x=1, upscale_y=1)

In [29]:
def _bin(
    start: float,
    stop: float,
    px_size: float,
    upscale: int,
) -> np.array:
    """Returns equally spaced points between start and stop, included.

    Args:
        start (float): Start point.
        stop (float): Stop point.
        px_size (float): Size of the pixels.
        upscale (int): Upscaling factor.

    Returns:
        output (npt.NDArray): Bin edges array.
    """
    return np.linspace(start, stop, int((stop - start) * upscale / px_size) + upscale)

In [ ]:
up_y, up_x = 1, 1

print(
    f"## ups_y = {up_y}, ups_x = {up_x}\n"
    f"len bins along y: {len(_bin(wfm.specs["mask_miny"], wfm.specs["mask_maxy"], wfm.specs["mask_deltay"], up_y))}\n"
    f"len bins along x: {len(_bin(wfm.specs["mask_minx"], wfm.specs["mask_maxx"], wfm.specs["mask_deltax"], up_x))}\n"
)

up_y, up_x = 5, 3

print(
    f"## ups_y = {up_y}, ups_x = {up_x}\n"
    f"len bins along y: {len(_bin(wfm.specs["mask_miny"], wfm.specs["mask_maxy"], wfm.specs["mask_deltay"], up_y))}\n"
    f"len bins along x: {len(_bin(wfm.specs["mask_minx"], wfm.specs["mask_maxx"], wfm.specs["mask_deltax"], up_x))}\n"
)

print(651*up_y, 1041*up_x)



## ups_y = 1, ups_x = 1
len bins along y: 651
len bins along x: 1041

## ups_y = 5, ups_x = 3
len bins along y: 3255
len bins along x: 3123

3255 3123


In [ ]:
def _bins_mask(
    self,
    upscale_f: UpscaleFactor,
) -> BinsRectangular:
    """Generate binning structure for mask with given upscale factors."""
    return BinsRectangular(
        _bin(self.mdl["mask_minx"], self.mdl["mask_maxx"], self.mdl["mask_deltax"] / upscale_f.x),
        _bin(self.mdl["mask_miny"], self.mdl["mask_maxy"], self.mdl["mask_deltay"] / upscale_f.y),
    )

def _bins_detector(self, upscale_f: UpscaleFactor) -> BinsRectangular:
    """Generate binning structure for detector with given upscale factors."""
    bins = self._bins_mask(self.upscale_f)
    xmin, xmax = _bisect_interval(bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
    ymin, ymax = _bisect_interval(bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
    return BinsRectangular(
        _bin(bins.x[xmin], bins.x[xmax], self.mdl["mask_deltax"] / upscale_f.x),
        _bin(bins.y[ymin], bins.y[ymax], self.mdl["mask_deltay"] / upscale_f.y),
    )

In [ ]:
import sys
import tempfile

import numpy as np
import numpy.typing as npt
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_pixel
from astropy.wcs.wcsapi import SlicedLowLevelWCS

from reproject.array_utils import iterate_chunks, sample_array_edges
from reproject.utils import parse_input_data, parse_output_projection
from reproject.mosaicking.subset_array import ReprojectedArraySubset

IS_WIN = sys.platform == "win32"


def sky_composition(
    input_data,
    output_projection,
    shape_out,
    reproject_function,
    combine_function,
) -> tuple[np.array, np.array]:
    
    # Parse the output projection to avoid having to do it for each
    wcs_out, shape_out = parse_output_projection(output_projection, shape_out=shape_out)

    output_array = np.zeros(shape_out)

    output_footprint = np.zeros(shape_out)

    on_the_fly = combine_function in ("mean", "sum")


    # Start off by reprojecting individual images to the final projection
    if not on_the_fly:
        arrays = []

    with tempfile.TemporaryDirectory(ignore_cleanup_errors=IS_WIN) as local_tmp_dir:
        for idata in range(len(input_data)):
            # We need to pre-parse the data here since we need to figure out how to
            # optimize/minimize the size of each output tile (see below).
            array_in, wcs_in = parse_input_data(input_data[idata], hdu_in=None)

            # Since we might be reprojecting small images into a large mosaic we
            # want to make sure that for each image we reproject to an array with
            # minimal footprint. We therefore find the pixel coordinates of the
            # edges of the initial image and transform this to pixel coordinates in
            # the final image to figure out the final WCS and shape to reproject to
            # for each tile. We strike a balance between transforming only the
            # input-image corners, which is fast but can cause clipping in cases of
            # significant distortion (when the edges of the input image become
            # convex in the output projection), and transforming every edge pixel,
            # which provides a lot of redundant information.
            edges = sample_array_edges(array_in.shape, n_samples=11)[::-1]
            edges_out = pixel_to_pixel(wcs_in, wcs_out, *edges)[::-1]

            # Determine the cutout parameters

            # In some cases, images might not have valid coordinates in the corners,
            # such as all-sky images or full solar disk views. In this case we skip
            # this step and just use the full output WCS for reprojection.
            ndim_out = len(shape_out)
            if np.any(np.isnan(edges_out)):
                bounds = list(zip([0] * ndim_out, shape_out, strict=False))
            else:
                bounds = []
                for idim in range(ndim_out):
                    imin = max(0, int(np.floor(edges_out[idim].min() + 0.5)))
                    imax = min(shape_out[idim], int(np.ceil(edges_out[idim].max() + 0.5)))
                    bounds.append((imin, imax))
                    if imax < imin: break

            slice_out = tuple([slice(imin, imax) for (imin, imax) in bounds])

            if isinstance(wcs_out, WCS):
                wcs_out_indiv = wcs_out[slice_out]
            else:
                wcs_out_indiv = SlicedLowLevelWCS(wcs_out.low_level_wcs, slice_out)

            shape_out_indiv = tuple([imax - imin for (imin, imax) in bounds])

            array = footprint = None

            array, footprint = reproject_function(
                (array_in, wcs_in),
                output_projection=wcs_out_indiv,
                shape_out=shape_out_indiv,
                hdu_in=None,
                output_array=array,
                output_footprint=footprint,
            )

            # For the purposes of mosaicking, we mask out NaN values from the array
            # and set the footprint to 0 at these locations.
            reset = np.isnan(array)
            array[reset] = 0.0
            footprint[reset] = 0.0

            array = ReprojectedArraySubset(array, footprint, bounds)

            if on_the_fly:
                # By default, values outside of the footprint are set to NaN
                # but we set these to 0 here to avoid getting NaNs in the
                # means/sums.
                array.array[array.footprint == 0] = 0
                output_footprint[array.view_in_original_array] += array.footprint
                # We now need to do output[view] += array * footprint but to avoid
                # the temporary array allocation from array * footprint we modify
                # array inplace, which we can do as the array will be discarded at
                # the end of the loop.
                array.array *= array.footprint
                output_array[array.view_in_original_array] += array.array

            else:
                arrays.append(array)


        if combine_function == "mean":
            with np.errstate(invalid="ignore"):
                output_array /= output_footprint

        if combine_function in ("first", "last", "min", "max"):
            if combine_function == "min":
                output_array[...] = np.inf
            elif combine_function == "max":
                output_array[...] = -np.inf

            for array in arrays:
                if combine_function == "first":
                    mask = output_footprint[array.view_in_original_array] == 0
                elif combine_function == "last":
                    mask = array.footprint > 0
                elif combine_function == "min":
                    mask = (array.footprint > 0) & (
                        array.array < output_array[array.view_in_original_array]
                    )
                elif combine_function == "max":
                    mask = (array.footprint > 0) & (
                        array.array > output_array[array.view_in_original_array]
                    )

                output_footprint[array.view_in_original_array] = np.where(
                    mask, array.footprint, output_footprint[array.view_in_original_array]
                )
                output_array[array.view_in_original_array] = np.where(
                    mask, array.array, output_array[array.view_in_original_array]
                )

    # We need to avoid potentially large memory allocation from output == 0 so
    # we operate in chunks.
    for chunk in iterate_chunks(output_array.shape, max_chunk_size=256 * 1024**2):
        output_array[chunk][output_footprint[chunk] == 0] = 0

    return output_array, output_footprint